<a href="https://colab.research.google.com/github/SheethHassan/AI-Based-Pneumonia-Detection/blob/ML-MODEL/Pneumonia_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#PYTHON LIBRARIES

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, callbacks, applications, models
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import (classification_report, confusion_matrix, f1_score, precision_score, recall_score, accuracy_score)

In [ ]:
# LOADING DATASET
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Verfing Folder Structure

import os
for split in ['train', 'val', 'test']:
  for cls in ['NORMAL', 'PNEUMONIA']:
    path = f'/content/drive/MyDrive/chest_xray/chest_xray/{split}/{cls}'
    count = len(os.listdir(path))
    print(f"{split}/{cls}:{count} images")

train/NORMAL:1342 images
train/PNEUMONIA:3876 images
val/NORMAL:9 images
val/PNEUMONIA:9 images
test/NORMAL:234 images
test/PNEUMONIA:390 images


In [ ]:
#Fix Valadation (since val is only 9 images)

base_dir = 'content/chest_xray/chest_xray'
train_dir = f'{base_dir}/train'
val_dir = f'{base_dir}/val'
test_dir = f'{base_dir}test'

In [ ]:
#Constants

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
CLASS_NAMES = ["NORMAL", "PNEUMONIA"]
INPUT_SHAPE = (224,224,3)
SEED = 42

In [ ]:

# Fix the directory paths
base_dir = '/content/drive/MyDrive/chest_xray/chest_xray'
train_dir = f'{base_dir}/train'
test_dir = f'{base_dir}/test'

#PREPROCESSING & AUGMENTATION
#AUGMENTATION ONLY FOR TRAINING DATA
train_datagen = ImageDataGenerator(
    rescale = 1./255,
    rotation_range = 20,
    width_shift_range = 0.1,
    height_shift_range = 0.1,
    shear_range = 0.15,
    zoom_range = 0.2,
    horizontal_flip = True,
    brightness_range = [0.8, 1.2],
    fill_mode = "nearest",
    validation_split = 0.20 #validation split
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    class_mode = 'binary',
    seed = SEED,
    shuffle = True,
    subset = 'training'
)
val_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    class_mode = "binary",
    seed = SEED,
    classes = CLASS_NAMES,
    subset = "validation"

)
test_gen = test_datagen.flow_from_directory(
    test_dir,
    target_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    class_mode = "binary",
    shuffle = False
)

print(f"Training Images: {train_gen.samples}")
print(f"validation Images:{val_gen.samples}")
print(f"Test Images: {test_gen.samples}")
print(f"Class indicies: {train_gen.class_indices}")

Found 4173 images belonging to 2 classes.
Found 1043 images belonging to 2 classes.
Found 624 images belonging to 2 classes.
Training Images: 4173
validation Images:1043
Test Images: 624
Class indicies: {'NORMAL': 0, 'PNEUMONIA': 1}


In [ ]:
#CLASS IMBALANCE
#Since the dataset has 1341 images train/NORMAL and 3875 images train/PNEUMONIA during traing without correction the model learns a bad habit which is guessing pneumonia 3x than guessing normal images which is called
# a biased model since it will miss real normal cases.
# This is fixed by using class imabalance
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight = 'balanced',
    classes = np.array([0,1]),
    y = train_gen.classes
)
class_weight_dict = {0:class_weights[0], 1:class_weights[1]}

print(f"Class weight for NORMAL is: {class_weight_dict[0]:.4f}")
print(f"Class weight for PNEUMONIA is : {class_weight_dict[1]:.4f}")

Class weight for NORMAL is: 1.9445
Class weight for PNEUMONIA is : 0.6731


In [ ]:
#MOBILENETv2 MODEL
#Load Mobilenetv2 pretrained on ImageNet
base_model = MobileNetV2(
    input_shape = INPUT_SHAPE,
    include_top = False, #Remove Imagenet classification head
    weights = "imagenet" #use pretrained weights
)

#Freeze Model

base_model.trainable = False


#Main Model
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation = 'relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid') #Binary output either PNEUMONIA OR NOT
])

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
# MODEL COMPILE

model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate = 0.0001),
    loss = 'binary_crossentropy',
    metrics = [
        'accuracy',
        tf.keras.metrics.AUC(name = 'auc'),
        tf.keras.metrics.Precision(name = 'precision'),
        tf.keras.metrics.Recall(name = 'recall')
    ]
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,435,393 (9.29 MB)

 Trainable params: 174,849 (683.00 KB)

 Non-trainable params: 2,260,544 (8.62 MB)

In [ ]:
#Callbacks
callbacks = [
    #stop learning if val_loss doesn't improve in 5 epochs
    tf.keras.callbacks.EarlyStopping(
        monitor = 'val_loss',
        patience = 5,
        restore_best_weights = True,
        verbose = 1
    ),

    #reduce learning rate if val_loss plateaus for 3 epochs
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor = 'val_loss',
        factor = 0.5,
        patience = 3,
        verbose = 1,
        min_lr = 0.0000001
    ),
    #save the best model automatically
    tf.keras.callbacks.ModelCheckpoint(
        filepath = '/content/drive/MyDrive/best_model.h5',
        monitor = 'val_auc',
        save_best_only = True,
        verbose = 1
    )
]

#Phase 1 -- Custom Top Layer Training
print("=" * 50)
print("Phase 1: Training Top layers of the model only")
print(f"Trainable params: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}")
print("=" * 50)

history_phase1 = model.fit(
    train_gen,
    validation_data = val_gen,
    epochs = 20,
    class_weight = class_weight_dict,
    callbacks = callbacks
)

Phase 1: Training Top layers of the model only
Trainable params: 174,849
Epoch 1/20
 31/131 ━━━━━━━━━━━━━━━━━━━━ 10:30 6s/step - accuracy: 0.5806 - auc: 0.5766 - loss: 0.7745 - precision: 0.7732 - recall: 0.6241

KeyboardInterrupt: 

In [ ]:
#PHASE 2 - Fine Tuning the Model
# Unfreeze Stage of top layers and training the rest of the model

print("=" * 50)
print("PHASE 2: Fine Tuning")
print("=" * 50)

#Unfreezing Model
base_model.trainable = True

#Freeze all layers except last 30 layers
for layer in base_model.layers[:-30]
layer.trainable = False


#Check how many layres are now trainable
trainble_layers = sum([1 for 1 in base_model.layers if l.trainable])
print(f"Trainable layers: {trainable_count}")

#Recompile with a much lower rate to increase accuracy a bit higher
model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate = 0.00001),
    loss = 'binary_crossentropy',
    metrics = [
        'accuracy',
        tf.keras.metrics.AUC(name = 'auc'),
        tf.keras.metrics.Precision(name = 'precision'),
        tf.keras.metrics.Recall(name = 'recall')
    ]
)

#update callbacks for phase 2
callbacks_phase2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor = 'val_loss',
        pateience = 5,
        restore_best_weights = True,
        verbose = 1
    ),
    tf.keras.calbacks.ReduceLROnPlateau(
        monitor = 'val_loss',
        factor = 0.5,
        patience = 3,
        verbose = 1,
        min_lr = 0.0000001
    ),
    #Save best model in google drive
    tf.keras.callbacks.ModelCheckpoint(
        filepath = 'content/drive/MyDrive/best_model.h5',
        monitor = 'val_auc',
        save_best_only = True,
        verbose = 1
    )
]

#Train Second Phase
history_phase2 = model.fit(
    train_gen,
    validation_data = val_gen,
    epochs = 20,
    class_weight = class_weight_dict,
    callbacks = callbacks_phase2
)

